# 개별종목 조합J — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합J 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합J의 피처 값만 지정합니다.
import json

COMBINATION = 'J'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합J 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5037,0.5012,0.0024,0.3049,0.3600,0.0805,0.3820,0.0761,0.1630
1,2,balanced,980,20150123,20150421,0.3932,0.3978,-0.0047,0.3502,0.3638,0.0586,0.3785,0.1994,0.2881
2,3,balanced,1210,20151228,20160328,0.3601,0.3762,-0.0161,0.3583,0.3580,0.0404,0.3720,0.3344,0.3505
3,4,balanced,1439,20161202,20170228,0.4631,0.4617,0.0014,0.3636,0.3834,0.1001,0.4098,0.1646,0.2731
4,5,balanced,1669,20171113,20180207,0.4221,0.3901,0.0320,0.3884,0.3990,0.1091,0.3977,0.2829,0.3538
5,6,balanced,1899,20181024,20190118,0.4121,0.3725,0.0396,0.4113,0.4206,0.1340,0.4183,0.5296,0.4447
6,7,balanced,2129,20190930,20191224,0.4706,0.4781,-0.0075,0.3550,0.3774,0.0944,0.4078,0.1705,0.2776
7,8,balanced,2359,20200902,20201130,0.4099,0.3476,0.0622,0.4061,0.4094,0.1149,0.4065,0.4257,0.4137
8,9,balanced,2589,20210806,20211105,0.3825,0.3914,-0.0089,0.3713,0.3827,0.0727,0.3840,0.2770,0.3364
9,10,balanced,2818,20220714,20221012,0.3533,0.3454,0.0079,0.3532,0.3564,0.0368,0.3696,0.2887,0.3288


,OOS 폴드 평균
accuracy,0.4140
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0172
macro_f1,0.3705
balanced_accuracy,0.3838
mcc,0.0867
pr_auc_macro_ovr,0.3943
down_recall,0.2812
core_harmonic_mean,0.3297


재실행 명령: python scripts/run_stock_model_experiment.py
